# 6 — GRU Preprocessing

**Purpose.** Merges the raw GPS / heart-rate / speed time series with the aerobic/anaerobic labels from `5_polarize.ipynb` and turns them into a fixed-size numeric tensor (workouts x 500 time steps x 4 channels: speed, heart rate, altitude, 1-D PCA of the GPS track). The data is split into train/test sets (stratified), scaled per channel with a `MinMaxScaler` fitted on the training set only, and saved to disk for the GRU tuning/training notebook.

**Inputs**
- `data/aerobik_anaerobik.csv` — labels and continuous score from `5_polarize.ipynb`
- `data/endomondoHR_speed.csv` — processed raw time series
- `functions.py` — helper functions (`parse_arr`)

**Outputs**
- `artifacts/gru_arrays.npz` — scaled train/test arrays and targets (`float32` features)
- `artifacts/gru_preprocessing.pkl` — channel order and fitted scalers (needed again at inference time)

**Note on labels.** The class labels `aerobik` ("aerobic") and `anaerobik` ("anaerobic") are intentionally Turkish-named; they are part of the project's API contract and are kept unchanged in code and data. Likewise, the QC reason codes (`cok_kisa`, `bozuk_timestamp`, `hiz_cok_eksik`, `hr_cok_eksik`, `cok_uzun`) are kept as-is.

Part of RunSight - see the repo README.

# GRU Preprocessing

**What does this notebook do?**
It combines the raw GPS / heart-rate / speed time series with the aerobic/anaerobic labels produced in `5_polarize.ipynb`, and converts them into a numeric tensor that a GRU (Gated Recurrent Unit) model can process directly.

**Where does it sit in the pipeline?**
`1_data_preprocess` -> `2_drop_columns` -> `3_haversine_savgol_filter` -> `4_Kmeans` (this approach was later abandoned) -> `5_polarize` (label generation: aerobic/anaerobic classification based on the personal heart-rate threshold) -> **`6_gru_preprocess`** -> `7_gru_tuner` (model setup and training).

**What problem does this notebook concretely solve?**
There are three problems: (1) matching the labels to the correct rows without losing track of which raw workout they belong to, (2) reducing the raw GPS coordinates to a single-dimensional movement signal that can be fed to the model, (3) converting each workout into a fixed-size array (4 channels x 500 time steps), splitting it into training/test sets and scaling it. Each section below corresponds to one of these problems.

In [ ]:
import os
import random, datetime
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

## 1. Load the labels

**Where do these labels come from?**
`aerobik_anaerobik.csv` was produced by the `run_zone_pipeline` function in `5_polarize.ipynb`. Each row corresponds to one workout and carries two target variables:

- `training_zone`: a binary label (`aerobik` / `anaerobik`), determined by a personalized threshold computed from the person's own heart-rate history.
- `anaerobic_time_frac`: a continuous score between 0 and 1 — it shows how much of the workout was spent above this threshold (0.05 means the workout was 95% aerobic and 5% anaerobic). This score is also kept because forcing mixed workouts such as "a short sprint inside an easy run" into a single class causes information loss.

**Why `index_col=0`?**
This column holds which row of the raw file (`endomondoHR_speed.csv`) each label belongs to. In the next step we will do the alignment over this index — if `index_col=0` is not given, this information turns into a column that looks like an ordinary row number, but is actually critical.

In [ ]:
labels_path = 'data/aerobik_anaerobik.csv'

labels = pd.read_csv(labels_path, index_col=0)
print(f"labels: {labels.shape}")
labels.head()

## 2. Pull the raw time series by index and merge

**Why don't we read the whole 2GB file?**
We only need the rows of `endomondoHR_speed.csv` that have a label. With the `skiprows` parameter, we check whether the `i`-th physical row of the CSV corresponds to the `(i-1)`-th index of `labels`, and skip the rows that do not match during reading — this uses much less RAM than loading the whole file into memory and filtering afterwards.

**Why do we merge with `join()` and not by column order?**
`labels.join(speed_df)` matches on the index value — it gives the correct result even if the row order of the two tables differs. A merge that relies on row order (like `pd.concat`) silently links the wrong rows together if the row order of the two files shifts at any point.

**What are the `assert`s right after the merge for?**
Both verify that the alignment is correct: whether the row counts are consistent, and whether any NaN appeared as a result of the merge. Their importance is this: if the index alignment breaks somehow (for example, if an index set from a different QC run is used), you can end up with a model trained on wrong labels without receiving any error message — the result runs but is meaningless. The asserts prevent this silent failure.

In [ ]:
PATH = 'data/endomondoHR_speed.csv'
# --- 1. Determine which original rows to pull ---
wanted_idx = set(labels.index)

# --- 2. Read ONLY these rows from the raw time-series file ---
# We use skiprows to avoid loading the whole 2GB file into memory:
# the i-th physical row of the CSV (header = row 0) -> original df index (i-1)
speed_df = pd.read_csv(
    PATH,
    index_col=0,
    skiprows=lambda i: i != 0 and (i - 1) not in wanted_idx,
)
print(f"speed_df (filtered): {speed_df.shape}")

# --- 3. Verify the alignment ---
assert len(speed_df) == len(labels), (
    f"Row counts do not match: speed_df={len(speed_df)}, labels={len(labels)}"
)
missing = wanted_idx - set(speed_df.index)
assert not missing, f"speed_df is missing indices: {sorted(missing)[:10]}"

# --- 4. Merge on the index ---
df_merged = labels.join(speed_df, how="left")
n_nan = df_merged.isna().sum().sum()
assert n_nan == 0, f"{n_nan} NaN values after the join, the alignment may be broken"

print(f"df_merged: {df_merged.shape[0]} rows, {df_merged.shape[1]} columns")
df_merged.head()

## 3. Channel preparation — which 4 signals we use and why

The array columns in `df_merged` are still text (that is how they came from the CSV) — we first convert all of them to numeric arrays. Since the dataset was already cleaned in the earlier steps (`1_data_preprocess`, `3_haversine_savgol_filter`), we do not do a separate NaN check / QC here.

**Which 4 time-series channels do we feed the GRU?**

- `speed`: speed derived from GPS with the haversine formula and smoothed with a Savitzky-Golay filter — it represents the pace of the workout.
- `heart_rate`: the direct physiological indicator of effort.
- `altitude`: elevation change. Why do we add it? At the same speed, the heart rate is higher on an uphill than on flat ground — looking only at speed and heart rate cannot distinguish this effect.
- `longitude`/`latitude` -> `latlon_pca`: instead of feeding the raw GPS coordinates directly, we reduce them to a single dimension that captures the net trend of movement (with the PCA step explained in the next section) and feed that.

**Why don't we give the model the person's personal heart-rate threshold (Karvonen HR reserve fraction)?**
It was deliberately left out. The aim is to test whether the model really learns effort patterns from the raw channels (speed, heart rate, altitude, position), rather than "memorizing" the label. If the threshold information were provided, the model could learn to look at whether the value is above/below the threshold and skip the time-series patterns it is actually supposed to learn.

In [ ]:
import functions as fc

num_cols = ['longitude', 'altitude', 'latitude', 'heart_rate', 'timestamp', 'speed']

for col in num_cols:
    df_merged[col] = df_merged[col].apply(fc.parse_arr)

lengths = df_merged["speed"].apply(len)
print("array lengths - min/max:", lengths.min(), lengths.max())

## 4. Lat/lon -> single dimension (PCA)

**Why don't we use latitude/longitude directly?**
1 degree of longitude corresponds to about 111 km at the equator and shrinks toward the poles — so the real-distance equivalent of longitude depends on the latitude (the same reasoning appeared in `3_haversine_savgol_filter.ipynb`). So we first convert each workout's coordinates into a local flat coordinate system (metres) relative to its first point (equirectangular approximation — gives a result close enough to haversine for short distances).

**What do we do next?**
With PCA(1), we project the transformed (x, y) points onto the single axis that carries the most variance. The result is a single number sequence that carries how far and in which (relative) "direction" the person moved net over the course of the workout.

**What is the limitation of this approach?**
The sign (direction) of the PCA can come out arbitrary — the `latlon_pca` values of two workouts on the same route may have opposite signs. Also, this channel carries only the net displacement trend, not the exact geometry of the route — we do not reconstruct the full GPS track. Since the GRU's purpose is not to draw a map but to use movement intensity in effort prediction, this is sufficient.

In [ ]:
from sklearn.decomposition import PCA

R_EARTH = 6371000.0  # metres


def latlon_to_local_xy(lat, lon):
    lat0, lon0 = lat[0], lon[0]
    lat0_rad = np.radians(lat0)
    x = R_EARTH * np.radians(lon - lon0) * np.cos(lat0_rad)
    y = R_EARTH * np.radians(lat - lat0)
    return x, y


def pca_1d_track(lat, lon):
    x, y = latlon_to_local_xy(lat, lon)
    xy = np.column_stack([x, y])
    proj = PCA(n_components=1).fit_transform(xy).ravel()
    return proj


df_merged["latlon_pca"] = [
    pca_1d_track(lat, lon)
    for lat, lon in zip(df_merged["latitude"], df_merged["longitude"])
]

## 5. Stack the channels, prepare the targets

**Why is `X_seq` shaped `(n_workouts, 500, 4)`?**
We stack the 4 channels (speed, heart_rate, altitude, latlon_pca) into a single 3-dimensional array. The GRU layer expects its input in exactly this format: 500 time steps x 4 features for each sample.

**Why are two separate targets (`y_binary`, `y_continuous`) kept?**
`y_binary` is `training_zone` converted to 0/1 — this is the main target the model is trained on. `y_continuous` is `anaerobic_time_frac`; it is not used in training, but is kept to check, after the model is trained, how well its output agrees with this continuous score (baseline comparison, `7_gru_tuner.ipynb`).

**What does the `np.bincount(y_binary)` output show?**
The distribution of the two classes (aerobic/anaerobic) in the dataset. If this distribution is imbalanced (one class clearly outnumbers the other), the model may be biased toward the majority class — this is exactly the reason `class_weight` is used in `7_gru_tuner.ipynb`.

In [ ]:
CHANNELS = ["speed", "heart_rate", "altitude", "latlon_pca"]

X_seq = np.stack(
    [np.column_stack([row[c] for c in CHANNELS]) for _, row in df_merged.iterrows()],
    axis=0,
)  # shape: (n_samples, 500, 4)

y_binary = (df_merged["training_zone"] == "anaerobik").astype(int).to_numpy()
y_continuous = df_merged["anaerobic_time_frac"].to_numpy()

print("X_seq shape:", X_seq.shape)
print("y_binary distribution (0=aerobik, 1=anaerobik):", np.bincount(y_binary))

## 6. Train/test split + per-channel scaling

**Why is the split by workout and not by user?**
The dataset (~70k workouts) does not have so few users that a single user could dominate — so the risk of the same person's different workouts falling into both train and test is not large enough to distort the results. With `stratify=y_binary`, what is actually cared about is ensured: both classes are represented in the same proportion in train and test.

**Why is scaling per channel and not with a single scaler?**
speed (km/h), heart_rate (bpm), altitude (metres) and latlon_pca (metres, but on a different scale) are in completely different units. Putting all of them through a single scaler would let the channel with the largest absolute values dominate the others. Instead, each channel is scaled separately into the [0, 1] range using its own min/max values.

**Why `MinMaxScaler` and not `StandardScaler`?**
In recurrent networks such as the GRU, having the inputs squeezed into a bounded range ([0, 1]) helps the internal `tanh`/`sigmoid` gates work properly and makes training more stable against exploding gradients.

**Why is the scaler `fit` only on the train set?**
Mixing the statistics (min/max) of the test set into the scaling would leak information from the test data to the model indirectly (data leakage) — this would be an advantage the model could never have in the real world. So `fit_transform` is applied only on train, and only `transform` on test.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import MinMaxScaler

X_train, X_test, y_train, y_test, yc_train, yc_test = train_test_split(
    X_seq, y_binary, y_continuous,
    test_size=0.2, random_state=46, stratify=y_binary,
)
print("train:", X_train.shape, "test:", X_test.shape)

n_channels = X_train.shape[2]
scalers = []
X_train_scaled = np.empty_like(X_train)
X_test_scaled = np.empty_like(X_test)

for c in range(n_channels):
    scaler = MinMaxScaler()
    X_train_scaled[:, :, c] = scaler.fit_transform(X_train[:, :, c])
    X_test_scaled[:, :, c] = scaler.transform(X_test[:, :, c])
    scalers.append(scaler)

print("scaling done. channel order:", CHANNELS)

## Save the preprocessing objects

The `scalers` and the channel order are needed so that the same transformation can be re-applied when the model is deployed (serverless inference) — they must be stored separately from the model file but together with it.

In [ ]:
import pickle

os.makedirs('artifacts', exist_ok=True)
with open('artifacts/gru_preprocessing.pkl', 'wb') as f:
    pickle.dump({"channel_order": CHANNELS, "scalers": scalers}, f)

print("preprocessing objects saved.")

## Save the training arrays to disk

The previous cell saved only the scalers — the actual `X_train_scaled` / `X_test_scaled` / `y_train` / `y_test` arrays stayed in this kernel. So that `7_gru_tuner.ipynb` can run in a separate kernel without needing this notebook's kernel, we also write the arrays to disk as `float32` (halves the space they take). After saving, you can shut down this notebook's kernel to free the RAM — notebook 7 inherits nothing from here, it reads everything from disk.

In [ ]:
np.savez(
    'artifacts/gru_arrays.npz',
    X_train_scaled=X_train_scaled.astype(np.float32),
    X_test_scaled=X_test_scaled.astype(np.float32),
    y_train=y_train,
    y_test=y_test,
    yc_train=yc_train,
    yc_test=yc_test,
)
print("arrays saved -> artifacts/gru_arrays.npz")